In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report
import sys
import os
import warnings
import random

warnings.filterwarnings('ignore')

# Append src directory to system path to import custom modules
sys.path.append(os.path.abspath('../src'))
from bearing_module import BearingCNN, predict_bearing_fault_real_time, class_names

# Plot configuration
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Load the dataset from the data directory
data = np.load('../data/CWRU_48k_load_1_CNN_data.npz')
X = data['data']
y = data['labels']

# Z-score normalization per sample
X_normalized = np.zeros_like(X)
for i in range(X.shape[0]):
    X_normalized[i] = (X[i] - X[i].mean()) / (X[i].std() + 1e-8)

# Encode string labels to integers
le = LabelEncoder()
y_encoded = le.fit_transform(y)
print(f"Detected Classes: {list(class_names)}")

# Split data (80% Train / 20% Test, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X_normalized, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# Convert to PyTorch tensors
X_train_t = torch.FloatTensor(X_train).unsqueeze(1) # Shape: (N, 1, 32, 32)
y_train_t = torch.LongTensor(y_train)
X_test_t = torch.FloatTensor(X_test).unsqueeze(1)
y_test_t = torch.LongTensor(y_test)

batch_size = 32
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=batch_size, shuffle=False)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Initialize model from the imported module
model = BearingCNN(num_classes=len(class_names)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

def run_epoch(model, loader, criterion, optimizer, device, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds = []
    context = torch.enable_grad() if train else torch.no_grad()

    with context:
        for batch_X, batch_y in loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            if train:
                optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += batch_y.size(0)
            correct += (predicted == batch_y).sum().item()
            all_preds.extend(predicted.cpu().numpy())

    return total_loss / len(loader), 100 * correct / total, all_preds

num_epochs = 50
history = {'train_loss': [], 'test_loss': [], 'train_acc': [], 'test_acc': []}

for epoch in range(num_epochs):
    train_loss, train_acc, _ = run_epoch(model, train_loader, criterion, optimizer, device, train=True)
    test_loss, test_acc, _ = run_epoch(model, test_loader, criterion, optimizer, device, train=False)
    scheduler.step(test_loss)

    history['train_loss'].append(train_loss)
    history['test_loss'].append(test_loss)
    history['train_acc'].append(train_acc)
    history['test_acc'].append(test_acc)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1:2d}/{num_epochs}] | Train Acc: {train_acc:.2f}% | Test Acc: {test_acc:.2f}%")

In [ ]:
# Final evaluation
_, final_acc, y_pred_all = run_epoch(model, test_loader, criterion, optimizer, device, train=False)
print(f"\nFinal Test Accuracy: {final_acc:.2f}%")

# 1. Plot and save Loss & Accuracy curves
epochs_range = range(1, num_epochs + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_range, history['train_loss'], label='Train Loss', color='steelblue')
axes[0].plot(epochs_range, history['test_loss'], label='Test Loss', color='tomato', linestyle='--')
axes[0].set_title('Train vs Test Loss')
axes[0].set_xlabel('Epochs')
axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].plot(epochs_range, history['train_acc'], label='Train Accuracy', color='steelblue')
axes[1].plot(epochs_range, history['test_acc'], label='Test Accuracy', color='tomato', linestyle='--')
axes[1].set_title('Train vs Test Accuracy')
axes[1].set_xlabel('Epochs')
axes[1].set_ylabel('Accuracy (%)')
axes[1].legend()

plt.tight_layout()
plt.savefig('../figures/train_vs_test_curves.png', dpi=300, bbox_inches='tight')
plt.show()

# 2. Plot and save Confusion Matrix
cm = confusion_matrix(y_test, y_pred_all)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('../figures/confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_all, target_names=class_names))

In [ ]:
# Save the trained model directly into the src folder for the dashboard to use
model_save_path = '../src/bearing_cnn_model.pth'
torch.save(model.state_dict(), model_save_path)
print(f"✅ Model saved successfully at: {os.path.abspath(model_save_path)}")

In [ ]:
# Select a random sample from the test set
idx = random.randint(0, len(X_test) - 1)
test_sample = X_test[idx]

# Hypothetical Health Index history (simulating a worsening trend)
hi_history = [15, 22, 35, 48, 67]

# Run the prediction pipeline (using the imported function from src/bearing_module.py)
result, probs = predict_bearing_fault_real_time(model, test_sample, hi_history, device)

print("\n--- Intelligent Fault Diagnosis Report ---")
print(f"Fault Type      : {result['Fault_Type']}")
print(f"Health Index    : {result['Health_Index']} / 100")
print(f"Confidence      : {result['Confidence']}%")
print(f"Risk Level      : {result['Risk_Level']} -> {result['State']}")
print(f"Urgency         : {result['Urgency']}")
print(f"Action          : {result['Recommended_Action']}")